# This notebook is to make base-difference and running-difference of SDO/AIA images

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import glob
import sunpy.map
from sunpy.time import parse_time
from astropy import units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import pandas as pd
# Show Plot in The Notebook 
import matplotlib
#matplotlib.use('nbagg')
# from aiapy.calibrate import normalize_exposure
from aiapy.calibrate import register, update_pointing
import astropy.io.fits as fits
import matplotlib.colors as colors
from astropy.visualization import ImageNormalize, PercentileInterval, SqrtStretch
import matplotlib.animation as animation

import sunpy.map
from sunpy.net import Fido, attrs as a

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

Removed the aiapy.calibrate.normalize_exposure function.
The same functionality can be achieved by dividing a `Map` by the exposure time property:
`my_map / my_map.exposure_time`

In [ ]:
# 2011-05-11, 2019-04-03, 2022-05-19
YEAR = '2021'
MONTH = '09'
DAY = '18'
start_hour = '00'
start_minute = '00'
end_hour = '23'
end_minute = '59'
channel = 211
local_dir = '/home/mnedal/data'

In [ ]:
# start_time = pd.Timestamp(f'{YEAR}-{MONTH}-{DAY} {start_hour}:{start_minute}:00')
# end_time = pd.Timestamp(f'{YEAR}-{MONTH}-{DAY} {end_hour}:{end_minute}:59')
# # end_time = start_time + datetime.timedelta(minutes=1)

# # Active region corona
# result211 = Fido.search(a.Time(start_time, end_time),
#                             a.Instrument('AIA'),
#                             a.Wavelength(channel*u.angstrom),
#                             a.Sample(1*u.min))
# aia211 = Fido.fetch(result211, path='/home/mnedal/data/{instrument}/{file}')

In [ ]:
# pathdir = '/Users/mnedal/DIAS/data'
pathdir = '/home/mnedal/data'
aia_fits = sorted(glob.glob(f'{pathdir}/AIA/*_211*.fits'))
len(aia_fits)

In [ ]:
aia_png = sorted(glob.glob(f'{local_dir}/plots/aia_sequence/211*.png'))
len(aia_png)

In [ ]:
aia_rd = sorted(glob.glob(f'{local_dir}/output_png/aia/RD/{YEAR}-{MONTH}-{DAY}*.png'))
len(aia_rd)

In [ ]:
m = sunpy.map.Map(aia_fits[0], vmin=0, vmax=2000)
m.peek()

In [ ]:
m.rsun_obs

In [ ]:
# exp_times = []
# for i, file in enumerate(aia_fits):
#     m = sunpy.map.Map(file)
#     exp_times.append(m.meta['exptime'])
# #     print(i, m.meta['exptime'])

In [ ]:
# np.max(exp_times), np.min(exp_times), np.mean(exp_times)

In [ ]:
# m.meta['exptime']

In [ ]:
# ignore images with exposure time less than a threshold
aia_maps = []
for file in aia_fits:
    m = sunpy.map.Map(file)
    if not m.meta['exptime'] < 2.9:
        aia_maps.append(m)

In [ ]:
len(aia_maps)

In [ ]:
m_seq = sunpy.map.Map(aia_maps, sequence=True)

# define a common normalization to use in the animation
for m in m_seq:
    m.plot_settings['norm'] = ImageNormalize(vmin=0,
                                             vmax=2e3,
                                             stretch=SqrtStretch())

In [ ]:
fig = plt.figure(figsize=[10,10])
ax = fig.add_subplot(111, projection=m_seq[0])
anim = m_seq.plot(axes=ax)
# fig.colorbar(anim)
ax.grid(False)
plt.show()

### Running-Difference 

In [ ]:
# make nested directories recursively
try:
    os.makedirs(f'{local_dir}/output_png/aia/RD/', exist_ok=True)
except:
    pass

In [ ]:
rundiff = [m - prev_m.quantity for m, prev_m in zip(m_seq[1:], m_seq[:-1])]
m_seq_running = sunpy.map.Map(rundiff, sequence=True)

# normalize the intensity ranges
low_thresh = -10
high_thresh = 10
norm = colors.Normalize(vmin=low_thresh, vmax=high_thresh)

for m in m_seq_running:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

In [ ]:
# Extract individual frames and export them as PNG images
for m in m_seq_running:
    fig = plt.figure(figsize=[5,5])
    ax = fig.add_subplot(111, projection=m_seq_running[0])
    m.plot()
    # m.draw_limb()
    # m.draw_grid()
    m.grid(False)
    fig.savefig(f"{local_dir}/output_png/aia/RD/{m.meta['date-obs']}.png", dpi=300, format='png', facecolor='w')
    plt.close()

### Export the sequence as GIF

In [ ]:
writergif = animation.PillowWriter(fps=10)
output = f"aia_{fits.open(aia_fits[0])[1].header['WAVE_STR']}"
anim.save(f'./{output}.gif', writer=writergif, savefig_kwargs={'facecolor':'w'})

### Export the sequence as MP4

In [ ]:
from matplotlib.animation import FFMpegWriter

# Define the writer for the video
writer = FFMpegWriter(fps=10, metadata=dict(artist='Me'), bitrate=1800)
output = f"aia_{fits.open(aia_fits[0])[1].header['WAVE_STR']}"
anim.save(f'./{output}.mp4', writer=writer)

### Export the sequence as PNG one by one and then make MP4 movie

In [ ]:
for m in m_seq:
    fig = plt.figure(figsize=[10,10])
    ax = fig.add_subplot(111, projection=m)
    img = m.plot(axes=ax)
    fig.colorbar(img, shrink=0.8, pad=0.02, label=m.meta['pixlunit'])
    ax.grid(False)
    fig.savefig(f"/home/mnedal/data/plots/aia_sequence/{m.meta['wave_str'].split('_')[0]}A_{m.meta['date-obs']}_.png",
                dpi=300, format='png', bbox_inches='tight')
    plt.close()

In [ ]:
from matplotlib.animation import FFMpegWriter

# Define the writer for the video
writer = FFMpegWriter(fps=10, metadata=dict(artist='Mohamed Nedal'), bitrate=1800)
output = f"aia_{fits.open(aia_fits[0])[1].header['WAVE_STR']}"
anim.save(f'{local_dir}/{output}.mp4', writer=writer)
print(f'Movie is exported successfully at: {local_dir}/{output}.mp4')

In [ ]:
from PIL import Image
from matplotlib.animation import FuncAnimation, FFMpegWriter


# Create a figure and axis
fig, ax = plt.subplots()

# Function to update the frame
def update(frame_number):
    img = Image.open(aia_rd[frame_number])
    ax.imshow(img)
    ax.axis('off')  # Hide axes
    return ax

# Define the writer for the video
writer = FFMpegWriter(fps=10, metadata=dict(artist='Mohamed Nedal'), bitrate=1800)

# Create the animation
ani = FuncAnimation(fig, update, frames=len(aia_rd), blit=False)

# Output file path
output = f"aia_{fits.open(aia_fits[0])[1].header['WAVE_STR']}.mp4"

# Save the animation
ani.save(f'{local_dir}/{output}', writer=writer)

print(f'Movie is exported successfully at: {local_dir}/{output}.mp4')

In [ ]:
1 + 1

In [ ]:
# m_seq = sunpy.map.Map(aia_maps, sequence=True)

# for m in m_seq:
#     m.plot_settings['norm'] = ImageNormalize(interval=PercentileInterval(90),
#                                              stretch=SqrtStretch(), clip=True)
# plt.figure(figsize=[10,10])
# anim = m_seq.plot()
# plt.show()

In [ ]:
# make base-difference sequence of images
m_seq_base = sunpy.map.Map([m - m_seq[0].quantity for m in m_seq[1:]], sequence=True)

In [ ]:
len(m_seq_base)

In [ ]:
low_thresh = -5
high_thresh = 3

norm = colors.Normalize(vmin=low_thresh, vmax=high_thresh)

for m in m_seq_base:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

In [ ]:
# drop these frames from the sequence
from sunpy.map import MapSequence

indices_to_drop = [2,4,16,18,20,22,24,26,28,30,32,34,36,38,40,42]
m_seq_base = MapSequence([m_seq_base[i] for i in range(len(m_seq_base)) if i not in indices_to_drop])

In [ ]:
plt.figure(figsize=[10,10])
anim_bd = m_seq_base.plot(title='Base Difference')
plt.show()

In [ ]:
anim_bd.save(f"{local_dir}/aia_BD_{fits.open(aia_fits[0])[1].header['WAVE_STR']}.gif", writer=writergif, savefig_kwargs={'facecolor':'w'})

In [ ]:
idx_list = [20, 23, 26, 29]
letters = ['A', 'B', 'C', 'D']

fig = plt.figure(figsize=[10,10])
for i, (idx, j) in enumerate(zip(idx_list, letters), start=1):
    print(i, idx, j)
    ax = fig.add_subplot(2,2,i, projection=m_seq_running[0])
    m_seq_running[idx].plot()
    m_seq_running[idx].draw_limb()
    m_seq_running[idx].draw_grid()
    ax.text(50, 3800, f'({j})', fontdict={'fontsize':'xx-large',
                                          'fontweight':'bold'}, color='white')
plt.tight_layout()
fig.savefig(f'{local_dir}/output_png/RD_panel.pdf', format='pdf', facecolor='w')
fig.savefig(f'{local_dir}/output_png/RD_panel.png', dpi=300, format='png', facecolor='w')
plt.show()

In [ ]:
len(m_seq_running)

In [ ]:
#for i in np.arange(16, len(m_seq_running), 5):
for i in np.arange(20, 31, 3):
#for i in np.arange(20, 21):
    print(f'Index: {i}')
    RD_frame(m_seq_running, i, channel=193, save=False)

In [ ]:
from sunpy.coordinates.utils import GreatArc

m = m_seq_running[32]

lon = [700, 500]
lat = [-600, -800]

start = SkyCoord(lon[0]*u.arcsec, lat[0]*u.arcsec, frame=m.coordinate_frame)
end = SkyCoord(lon[1]*u.arcsec, lat[1]*u.arcsec, frame=m.coordinate_frame)
great_arc = GreatArc(start, end)

fig = plt.figure()
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
ax.plot_coord(great_arc.coordinates(), color='red')
plt.show()

In [ ]:
plt.figure(figsize=(7,7))
anim_rd = m_seq_running.plot(title='Running Difference')
plt.show()

In [ ]:
anim_rd.save(local_dir+'aia_RD_{}.gif'.format(fits.open(aia_fits[0])[1].header['WAVE_STR']), writer=writergif, savefig_kwargs={'facecolor':'w'})

In [ ]:
plt.figure(figsize=(10,10))
m_seq_base[67].plot()
plt.show()

In [ ]:
help(a.Instrument.xrs)

In [ ]:
dir(a.Instrument)

In [ ]:
tstart = '2019-04-03 12:00'
tend   = '2019-04-03 13:00'
result = Fido.search(a.Time(tstart, tend), 
                     a.Instrument.xrs
                    )
print(result)

### Making RGB image

In [ ]:
local_dir = f'/SSD/{YEAR}{MONTH}{DAY}/AIA'

aia171_fits = sorted(glob.glob(f'{local_dir}/171/*.fits'))
aia193_fits = sorted(glob.glob(f'{local_dir}/193/*.fits'))
aia211_fits = sorted(glob.glob(f'{local_dir}/211/*.fits'))

aia171_maps = []
aia193_maps = []
aia211_maps = []

for file in aia171_fits:
    m = sunpy.map.Map(file)
    aia171_maps.append(m)

for file in aia193_fits:
    m = sunpy.map.Map(file)
    aia193_maps.append(m)

for file in aia211_fits:
    m = sunpy.map.Map(file)
    aia211_maps.append(m)

m171_seq = sunpy.map.Map(aia171_maps, sequence=True)
m193_seq = sunpy.map.Map(aia193_maps, sequence=True)
m211_seq = sunpy.map.Map(aia211_maps, sequence=True)

for m in m171_seq:
    m.plot_settings['norm'] = ImageNormalize(vmin=0, 
                                             vmax=4e3, 
                                             stretch=SqrtStretch())
for m in m193_seq:
    m.plot_settings['norm'] = ImageNormalize(vmin=0, 
                                             vmax=4e3, 
                                             stretch=SqrtStretch())
for m in m211_seq:
    m.plot_settings['norm'] = ImageNormalize(vmin=0, 
                                             vmax=4e3, 
                                             stretch=SqrtStretch())

# ### This next part is case-sensitive
# # drop these frames from the sequence
# from sunpy.map import MapSequence

# indices_to_drop = [2,4,16,18,20,22,24,26,28,30,32,34,36,38,40,42]

# m171_seq = MapSequence([m171_seq[i] for i in range(len(m171_seq)) if i not in indices_to_drop])
# m193_seq = MapSequence([m193_seq[i] for i in range(len(m193_seq)) if i not in indices_to_drop])
# m211_seq = MapSequence([m211_seq[i] for i in range(len(m211_seq)) if i not in indices_to_drop])



m171_seq_running = sunpy.map.Map([m - prev_m.quantity for m, prev_m in zip(m171_seq[1:], m171_seq[:-1])], sequence=True)
m193_seq_running = sunpy.map.Map([m - prev_m.quantity for m, prev_m in zip(m193_seq[1:], m193_seq[:-1])], sequence=True)
m211_seq_running = sunpy.map.Map([m - prev_m.quantity for m, prev_m in zip(m211_seq[1:], m211_seq[:-1])], sequence=True)

low_thresh = -10
high_thresh = 10
norm = colors.Normalize(vmin=low_thresh, vmax=high_thresh)

for m in m171_seq_running:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

for m in m193_seq_running:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

for m in m211_seq_running:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

In [ ]:
idx = -1
fig = plt.figure(figsize=[15,5])

ax = fig.add_subplot(131, projection=m171_seq[idx])
m171_seq[idx].plot(axes=ax)

ax = fig.add_subplot(132, projection=m193_seq[idx])
m193_seq[idx].plot(axes=ax)

ax = fig.add_subplot(133, projection=m211_seq[idx])
m211_seq[idx].plot(axes=ax)

plt.tight_layout()
plt.show()

In [ ]:
idx = 34
fig = plt.figure(figsize=[15,5])

ax = fig.add_subplot(131, projection=m171_seq_running[idx])
m171_seq_running[idx].plot(axes=ax)

ax = fig.add_subplot(132, projection=m193_seq_running[idx])
m193_seq_running[idx].plot(axes=ax)

ax = fig.add_subplot(133, projection=m211_seq_running[idx])
m211_seq_running[idx].plot(axes=ax)

plt.tight_layout()
plt.show()

In [ ]:
def round_obstime(time=None):
    '''
    Round the observation time to put it in the image title.
    '''
    from datetime import datetime, timedelta

    original_time_str = time

    # Convert the original time string to a datetime object
    original_time = datetime.strptime(original_time_str, '%H:%M:%S.%f')

    # Round the time to the nearest second
    rounded_time = original_time + timedelta(seconds=round(original_time.microsecond / 1e6))

    # Format the rounded time as 'HH:MM:SS'
    rounded_time_str = rounded_time.strftime('%H:%M:%S')
    return rounded_time_str

In [ ]:
date_str = str(m171_seq_running[idx].date).split('T')[0]
rounded_time_str = round_obstime(time=str(m171_seq_running[idx].date).split('T')[1])
f'{date_str} {rounded_time_str}'

In [ ]:
# # Normalize values to be in the range [0,1]
# red_channel_normalized = m171_seq_running[idx].data / np.max(m171_seq_running[idx].data)
# green_channel_normalized = m193_seq_running[idx].data / np.max(m193_seq_running[idx].data)
# blue_channel_normalized = m211_seq_running[idx].data / np.max(m211_seq_running[idx].data)

# Stack the channels to create an RGB image
#rgb_image = np.stack([red_channel_normalized, green_channel_normalized, blue_channel_normalized], axis=-1)

idx = 2
rgb_image = np.stack([m171_seq_running[idx].data, m193_seq_running[idx].data, m211_seq_running[idx].data], axis=-1)

# Display the RGB image
fig = plt.figure(figsize=[5,5])
ax = fig.add_subplot(111, projection=m171_seq_running[idx])
ax.imshow(rgb_image)
ax.set_xlabel('Helioprojective Longitude (arcsec)')
ax.set_ylabel('Helioprojective Latitude (arcsec)')

date_str = str(m171_seq_running[idx].date).split('T')[0]
rounded_time_str = round_obstime(time=str(m171_seq_running[idx].date).split('T')[1])
ax.set_title(f'{date_str} {rounded_time_str}')

# Create custom legend text labels with different colors
legend_labels = [
    ('AIA 171 $\AA$ Channel', 'red'),
    ('AIA 193 $\AA$ Channel', 'lime'),
    ('AIA 211 $\AA$ Channel', 'blue')
]

# Add custom legend to the plot
for label, color in legend_labels:
    ax.text(0.03, 0.95 - 0.03*legend_labels.index((label, color)), label, color=color, backgroundcolor='gray', transform=ax.transAxes, fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Create custom legend text labels with different colors
legend_labels = [
    ('AIA 171 $\AA$ Channel', 'red'),
    ('AIA 193 $\AA$ Channel', 'lime'),
    ('AIA 211 $\AA$ Channel', 'blue')
]
idx_list = [20, 23, 26, 29]
letters = ['A', 'B', 'C', 'D']

fig = plt.figure(figsize=[10,10])
for i, (idx, j) in enumerate(zip(idx_list, letters), start=1):
    ax = fig.add_subplot(2,2,i, projection=m171_seq_running[0])
    rgb_image = np.stack([m171_seq_running[idx].data, m193_seq_running[idx].data, m211_seq_running[idx].data], axis=-1)
    ax.imshow(rgb_image)
    ax.set_xlabel('Helioprojective Longitude (arcsec)')
    ax.set_ylabel('Helioprojective Latitude (arcsec)')
    
    m171_seq_running[idx].draw_limb()
    m171_seq_running[idx].draw_grid()
    
    date_str = str(m171_seq_running[idx].date).split('T')[0]
    rounded_time_str = round_obstime(time=str(m171_seq_running[idx].date).split('T')[1])
    ax.set_title(f'{date_str} {rounded_time_str}')
    
    ax.text(50, 3800, f'({j})', fontdict={'fontsize':'xx-large',
                                          'fontweight':'bold'}, color='white')
    # Add custom legend to the plot
    for label, color in legend_labels:
        ax.text(0.03, 0.03 + 0.05*legend_labels.index((label, color)), label, color=color, backgroundcolor='gray', transform=ax.transAxes, fontsize=8)

plt.tight_layout()
fig.savefig(f'{local_dir}/output_png/RGB_panel.pdf', format='pdf', facecolor='w')
fig.savefig(f'{local_dir}/output_png/RGB_panel.png', dpi=300, format='png', facecolor='w')
plt.show()

### Make a GIF animation

In [ ]:
os.makedirs(f'{local_dir}/RGB/', exist_ok=True)

# in case they have different lengths
for idx in range(min(len(m171_seq_running), len(m193_seq_running), len(m211_seq_running))):
    
    # Stack the channels to create an RGB image
    rgb_image = np.stack([m171_seq_running[idx].data, m193_seq_running[idx].data, m211_seq_running[idx].data], axis=-1)
    
    # Display the RGB image
    fig = plt.figure(figsize=[10,10])
    ax = fig.add_subplot(111, projection=m171_seq_running[idx])
    ax.imshow(rgb_image)
    ax.set_xlabel('Helioprojective Longitude (arcsec)')
    ax.set_ylabel('Helioprojective Latitude (arcsec)')
    
    # Create custom legend text labels with different colors
    legend_labels = [
        ('AIA 171 $\AA$ Channel', 'red'),
        ('AIA 193 $\AA$ Channel', 'lime'),
        ('AIA 211 $\AA$ Channel', 'blue')
    ]
    
    # Add custom legend to the plot
    for label, color in legend_labels:
        ax.text(0.03, 0.95 - 0.03*legend_labels.index((label, color)), label, color=color, backgroundcolor='gray', transform=ax.transAxes, fontsize=12)
    
    plt.tight_layout()
    
    if idx < 10:
        image_name = f'aia_rgb0{idx}'
    else:
        image_name = f'aia_rgb{idx}'
    
    fig.savefig(f'{local_dir}/RGB/{image_name}.png', format='png', dpi=100, bbox_inches='tight')
    plt.ioff()

In [ ]:
import imageio
from skimage.transform import resize

# List all PNG files in the folder
image_files = sorted(glob.glob(f'{local_dir}/RGB/*.png'))

# Create a list to store images
images = []

# Read each image and append to the list
for image_file in image_files:
    img = imageio.imread(image_file)
    img = (img.astype(np.float32) / 255.0).clip(0, 1)  # Convert to [0, 1] range
    images.append((img * 255).astype(np.uint8))  # Convert to uint8

# Resize images to a smaller resolution (adjust as needed)
target_resolution = (512, 512)
images = [resize(img, target_resolution, preserve_range=True).astype(np.uint8) for img in images]

# Save the images as a GIF animation
imageio.mimsave(f'{local_dir}/RGB/output.gif', images, duration=0.2, loop=0)

In [ ]:
### Full resolution
import imageio

# List all PNG files in the folder
image_files = sorted(glob.glob(f'{local_dir}/RGB/*.png'))

# Create a list to store images
images = []

# Read each image and append to the list
for image_file in image_files:
    img = imageio.imread(image_file)
    images.append(img)

# Save the images as a GIF animation
imageio.mimsave(f'{local_dir}/RGB/output.gif', images, duration=0.2, loop=0)

In [ ]:
local_dir

In [ ]:
for idx in range(len(m171_seq_running)):
    rgb_image = np.stack([m171_seq_running[idx].data, m193_seq_running[idx].data, m211_seq_running[idx].data], axis=-1)
    
    fig = plt.figure(figsize=[5,5])
    ax = fig.add_subplot(111, projection=m171_seq_running[idx])
    ax.imshow(rgb_image)
    ax.set_title(f'Image: {idx}')
    ax.set_xlabel('Helioprojective Longitude (arcsec)')
    ax.set_ylabel('Helioprojective Latitude (arcsec)')

    plt.tight_layout()
    plt.show()

In [ ]:
from aiapy.calibrate import normalize_exposure


local_dir = f'/SSD/{YEAR}{MONTH}{DAY}/AIA'

aia171_fits = sorted(glob.glob(f'{local_dir}/171/*.fits'))
aia193_fits = sorted(glob.glob(f'{local_dir}/193/*.fits'))
aia211_fits = sorted(glob.glob(f'{local_dir}/211/*.fits'))

aia171_maps = []
aia193_maps = []
aia211_maps = []

for file in aia171_fits:
    m = sunpy.map.Map(file)
    aia171_maps.append(m)

for file in aia193_fits:
    m = sunpy.map.Map(file)
    aia193_maps.append(m)

for file in aia211_fits:
    m = sunpy.map.Map(file)
    aia211_maps.append(m)

m171_seq = sunpy.map.Map(aia171_maps, sequence=True)
m193_seq = sunpy.map.Map(aia193_maps, sequence=True)
m211_seq = sunpy.map.Map(aia211_maps, sequence=True)

m171_normalized = MapSequence([normalize_exposure(m) for m in m171_seq])
m193_normalized = MapSequence([normalize_exposure(m) for m in m193_seq])
m211_normalized = MapSequence([normalize_exposure(m) for m in m211_seq])

# make base-difference sequence of images
m171_seq_base = sunpy.map.Map([m - m171_normalized[0].quantity for m in m171_normalized[1:]], sequence=True)
m193_seq_base = sunpy.map.Map([m - m193_normalized[0].quantity for m in m193_normalized[1:]], sequence=True)
m211_seq_base = sunpy.map.Map([m - m211_normalized[0].quantity for m in m211_normalized[1:]], sequence=True)

# make running-difference sequence of images
m171_seq_running = sunpy.map.Map([m - prev_m.quantity for m, prev_m in zip(m171_normalized[1:], m171_normalized[:-1])], sequence=True)
m193_seq_running = sunpy.map.Map([m - prev_m.quantity for m, prev_m in zip(m193_normalized[1:], m193_normalized[:-1])], sequence=True)
m211_seq_running = sunpy.map.Map([m - prev_m.quantity for m, prev_m in zip(m211_normalized[1:], m211_normalized[:-1])], sequence=True)

low_thresh = -10
high_thresh = 10
norm = colors.Normalize(vmin=low_thresh, vmax=high_thresh)

# for base-difference images
for m in m171_seq_base:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

for m in m193_seq_base:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

for m in m211_seq_base:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

# for running-difference images
for m in m171_seq_running:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

for m in m193_seq_running:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

for m in m211_seq_running:
    m.plot_settings['norm'] = norm
    m.plot_settings['cmap'] = 'Greys_r'

In [ ]:
# Create custom legend text labels with different colors
legend_labels = [
    ('AIA 171 $\AA$ Channel', 'red'),
    ('AIA 193 $\AA$ Channel', 'lime'),
    ('AIA 211 $\AA$ Channel', 'blue')
]
idx_list = [17, 20, 23, 26]
letters = ['A', 'B', 'C', 'D']

fig = plt.figure(figsize=[10,10])
for i, (idx, j) in enumerate(zip(idx_list, letters), start=1):
    ax = fig.add_subplot(2,2,i, projection=m171_seq_running[0])
    rgb_image = np.stack([m171_seq_running[idx].data, m193_seq_running[idx].data, m211_seq_running[idx].data], axis=-1)
    ax.imshow(rgb_image)
    ax.set_xlabel('Helioprojective Longitude (arcsec)')
    ax.set_ylabel('Helioprojective Latitude (arcsec)')
    
    m171_seq_running[idx].draw_limb()
    m171_seq_running[idx].draw_grid()
    
    date_str = str(m171_seq_running[idx].date).split('T')[0]
    rounded_time_str = round_obstime(time=str(m171_seq_running[idx].date).split('T')[1])
    ax.set_title(f'{date_str} {rounded_time_str}')
    
    ax.text(50, 3800, f'({j})', fontdict={'fontsize':'xx-large',
                                          'fontweight':'bold'}, color='white')
    # Add custom legend to the plot
    for label, color in legend_labels:
        ax.text(0.03, 0.03 + 0.05*legend_labels.index((label, color)), label, color=color, backgroundcolor='gray', transform=ax.transAxes, fontsize=8)

plt.tight_layout()
fig.savefig(f'{local_dir}/output_png/RGB_panel.png', dpi=300, format='png', facecolor='w')
plt.show()

In [ ]:
# base-difference
for idx in range(len(m171_seq_base)):
    rgb_image = np.stack([m171_seq_base[idx].data, m193_seq_base[idx].data, m211_seq_base[idx].data], axis=-1)
    
    fig = plt.figure(figsize=[5,5])
    ax = fig.add_subplot(111, projection=m171_seq_base[idx])
    ax.imshow(rgb_image)
    ax.set_title(f'Image: {idx}')
    ax.set_xlabel('Helioprojective Longitude (arcsec)')
    ax.set_ylabel('Helioprojective Latitude (arcsec)')
    plt.show()